# 11 — Génération de la soumission Kaggle

**Objectif :** produire une colonne `sku` contenant cinq SKU séparés par des espaces.

**Entrées :** test nettoyé et bundle de production.  
**Sorties :** `submissions/predictions.csv` et rapport de validation.  
**Dépendance :** notebook 10.  
**Temps estimé :** dépend du nombre de requêtes ; l'encodage utilise CUDA.  
**Ressources :** GPU recommandé, fallback CPU.

In [ ]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import pandas as pd


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Racine du projet introuvable.")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.recommender import HybridRecommender

PROCESSED_DIR = ROOT / "data" / "processed"
SUBMISSIONS_DIR = ROOT / "submissions"
REPORTS_DIR = ROOT / "reports"
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

test = pd.read_csv(PROCESSED_DIR / "test_clean.csv").fillna("")
recommender = HybridRecommender(ROOT / "artifacts")

## Inférence

In [ ]:
started = time.perf_counter()
prediction_rows: list[str] = []
for query in test["query"].astype(str):
    skus = [row["sku"] for row in recommender.recommend(query, k=5)]
    if len(skus) != 5 or len(set(skus)) != 5:
        raise ValueError(f"Top 5 invalide pour la requête {query!r}: {skus}")
    prediction_rows.append(" ".join(skus))

submission = pd.DataFrame({"sku": prediction_rows})
output_path = SUBMISSIONS_DIR / "predictions.csv"
submission.to_csv(output_path, index=False)
elapsed = time.perf_counter() - started
display(submission.head())

## Validation stricte

In [ ]:
reloaded = pd.read_csv(output_path, dtype=str, keep_default_na=False)
assert reloaded.columns.tolist() == ["sku"]
assert len(reloaded) == len(test)
assert reloaded["sku"].map(lambda value: len(value.split()) == 5).all()
assert reloaded["sku"].map(lambda value: len(set(value.split())) == 5).all()

source = json.loads((REPORTS_DIR / "data_inventory.json").read_text(encoding="utf-8"))["source"]
report = {
    "status": "valid",
    "source": source,
    "rows": len(reloaded),
    "columns": reloaded.columns.tolist(),
    "seconds": elapsed,
    "eligible_for_kaggle": source == "kaggle",
    "output": str(output_path.relative_to(ROOT)),
}
(REPORTS_DIR / "submission_validation.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(json.dumps(report, indent=2, ensure_ascii=False))

## Conclusion

Le fichier suit le format de la baseline officielle : une colonne `sku`, avec cinq identifiants
séparés par des espaces. Une soumission synthétique est validée techniquement mais reste inéligible.